# GNN Bipartite Classification

Reproduction of §5 of Bescos/Moyal/Veyrat (Stanford 2019): **GraphSAGE** and **GAT** on the Δ-graph to classify stocks (above/below median next-quarter log-return) and investors (above/below median next-quarter profitability).

- Nodes: CIKs ∪ CUSIPs
- Edges: Δ-weights from `changed_holdings` (quarter t)
- Targets: binary labels from `stocks_return` at quarter t+1 and investor profitability at t+1
- Two eval modes: **node-mask** (random 20% holdout on same graph) and **two-periods** (train on Δ_t, test on Δ_{t+1})

## Required `cusip_financial_data` schema

This notebook assumes a table with one row per (cusip, year, quarter) providing the **intensive** financial ratios from the paper. When the table isn't yet populated the notebook falls back to zero-filled features so you can wire everything up end-to-end first.

```sql
CREATE TABLE cusip_financial_data (
    cusip           TEXT    NOT NULL,
    year            SMALLINT NOT NULL,
    quarter         SMALLINT NOT NULL,
    diluted_eps     DOUBLE PRECISION,   -- diluted earnings per share
    roe             DOUBLE PRECISION,   -- return on equity
    ev_ebitda       DOUBLE PRECISION,   -- enterprise value / EBITDA
    pe_ratio        DOUBLE PRECISION,   -- price / earnings
    price_to_sales  DOUBLE PRECISION,
    price_to_book   DOUBLE PRECISION,
    debt_to_equity  DOUBLE PRECISION,
    dividend_yield  DOUBLE PRECISION,
    fcf_per_share   DOUBLE PRECISION,   -- free cash flow per share
    PRIMARY KEY (cusip, year, quarter)
);
```

`log_return` for the same quarter is already in `stocks_return` and is appended automatically.

All features are intensive (ratios / per-share) so they're comparable across different market caps — same rationale as the paper.

## Setup

In [ ]:
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv, GATConv

warnings.filterwarnings("ignore", category=UserWarning)

for env_path in [Path.cwd() / ".env", Path.cwd().parent / ".env", Path.cwd().parent.parent / ".env"]:
    if env_path.exists():
        load_dotenv(env_path)
        break

def project_root() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd, cwd.parent, cwd.parent.parent]:
        if (p / "ETL").is_dir():
            return p
    return cwd.parent.parent

ROOT = project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from ETL.gnn_db_pipeline.config import (
    TARGET_DB,
    TGT_CHANGED_HOLDINGS,
    TGT_STOCKS_RETURN,
    TGT_CIK_AUM,
    TGT_NORMALIZED_HOLDINGS,
)
from ETL.gnn_db_pipeline.db_connector import ConfigurablePostgresHandler

CUSIP_FIN_TABLE = "cusip_financial_data"
STOCK_FEATURE_COLS = [
    "diluted_eps", "roe", "ev_ebitda", "pe_ratio", "price_to_sales",
    "price_to_book", "debt_to_equity", "dividend_yield", "fcf_per_share",
    "log_return",
]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.05)
torch.manual_seed(0); np.random.seed(0)

handler = ConfigurablePostgresHandler(TARGET_DB)
handler.connect()
print("Connected to", TARGET_DB, "| device", DEVICE)

## Config

In [ ]:
# Target period for single-graph evaluation
YEAR, QUARTER = 2016, 3

# Training
HIDDEN_DIM = 32
NUM_LAYERS = 2
EPOCHS = 100
LR = 8e-4
DROPOUT = 0.5
MASK_FRAC = 0.20    # node-mask eval holdout
GAT_HEADS = 4

## Data loaders

In [ ]:
def next_year_quarter(year: int, quarter: int):
    return (year + 1, 1) if quarter == 4 else (year, quarter + 1)

def load_edges(year, quarter):
    return handler.query(f"""
        SELECT cik, cusip, change_in_weight AS w
        FROM {TGT_CHANGED_HOLDINGS}
        WHERE year = %s AND quarter = %s AND change_in_weight IS NOT NULL
    """, (year, quarter))

def load_returns(year, quarter):
    return handler.query(f"""
        SELECT cusip, log_return FROM {TGT_STOCKS_RETURN}
        WHERE year = %s AND quarter = %s AND log_return IS NOT NULL
    """, (year, quarter))

def load_aum(year, quarter):
    return handler.query(f"""
        SELECT cik, total AS aum FROM {TGT_CIK_AUM}
        WHERE year = %s AND quarter = %s AND total > 0
    """, (year, quarter))

def load_stock_features(year, quarter) -> pd.DataFrame:
    """Pull intensive financial ratios + log_return for the quarter. Returns empty if table missing."""
    try:
        fin = handler.query(f"""
            SELECT * FROM {CUSIP_FIN_TABLE}
            WHERE year = %s AND quarter = %s
        """, (year, quarter))
    except Exception as e:
        print(f"  [warn] {CUSIP_FIN_TABLE} not available ({e.__class__.__name__}); zero-filling features")
        fin = pd.DataFrame(columns=["cusip"] + STOCK_FEATURE_COLS)
    rets = load_returns(year, quarter)
    df = fin.merge(rets, on="cusip", how="outer")
    for c in STOCK_FEATURE_COLS:
        if c not in df.columns:
            df[c] = 0.0
    return df[["cusip"] + STOCK_FEATURE_COLS]

def investor_profitability(year, quarter) -> pd.Series:
    """Investor profitability at period t: Σ w_i(t) · log_return_i(t+1).
    Requires weights at t from normalized_holdings and returns at t+1."""
    ny, nq = next_year_quarter(year, quarter)
    h = handler.query(f"""
        SELECT cik, cusip, weight FROM {TGT_NORMALIZED_HOLDINGS}
        WHERE year = %s AND quarter = %s
    """, (year, quarter))
    r = load_returns(ny, nq)
    m = h.merge(r, on="cusip", how="inner")
    m["contrib"] = m["weight"] * m["log_return"]
    return m.groupby("cik")["contrib"].sum().rename("profitability")

## Graph builder

Build a single `torch_geometric.data.Data` object for one quarter: uniform-sized node tensors (paper trick), edge-weighted undirected edges, and binary labels defined by median split on stock returns / investor profitability at t+1.

In [ ]:
def zscore(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    out = df.copy()
    for c in cols:
        v = out[c].astype(float)
        v = v.replace([np.inf, -np.inf], np.nan).fillna(v.median() if v.notna().any() else 0.0)
        sd = v.std()
        out[c] = (v - v.mean()) / sd if sd > 0 else 0.0
    return out

def build_graph(year: int, quarter: int):
    """Build one bipartite Δ-graph with node features and binary labels.
    Returns (Data, meta) where meta holds idx ↔ id mappings and label/type masks."""
    edges = load_edges(year, quarter)
    if edges.empty:
        raise RuntimeError(f"no Δ-edges for {year} Q{quarter}")
    ny, nq = next_year_quarter(year, quarter)

    # CIK features: portfolio size + n_holdings + past-quarter profitability (from t-1→t)
    aum = load_aum(year, quarter)
    py, pq = (year - 1, 4) if quarter == 1 else (year, quarter - 1)
    try:
        past_prof = investor_profitability(py, pq).reset_index()
    except Exception:
        past_prof = pd.DataFrame(columns=["cik", "profitability"])
    cik_nh = edges.groupby("cik").size().rename("n_holdings").reset_index()
    cik_df = aum.merge(cik_nh, on="cik", how="outer").merge(past_prof, on="cik", how="left")
    cik_df["aum"] = cik_df["aum"].fillna(cik_df["aum"].median() if cik_df["aum"].notna().any() else 0.0)
    cik_df["log_aum"] = np.log(cik_df["aum"].clip(lower=1.0))
    cik_df["n_holdings"] = cik_df["n_holdings"].fillna(0)
    cik_df["profitability"] = cik_df["profitability"].fillna(0.0)
    CIK_FEATS = ["log_aum", "n_holdings", "profitability"]
    cik_df = zscore(cik_df, CIK_FEATS)

    # Stock features
    stock_df = load_stock_features(year, quarter)
    stock_df = zscore(stock_df, STOCK_FEATURE_COLS)

    # Labels at t+1
    r_next = load_returns(ny, nq).set_index("cusip")["log_return"]
    prof_next = investor_profitability(year, quarter)  # uses t weights + t+1 returns

    # Restrict to nodes seen in the edge list
    cik_ids = pd.Index(edges["cik"].unique())
    cusip_ids = pd.Index(edges["cusip"].unique())

    # Build feature matrices aligned to node order: [cik..., cusip...]
    cik_df = cik_df.set_index("cik").reindex(cik_ids).fillna(0.0)
    stock_df = stock_df.set_index("cusip").reindex(cusip_ids).fillna(0.0)

    # Uniform feature dim: pad the smaller tensor with zero columns (paper's trick)
    F_cik, F_stk = cik_df[CIK_FEATS].to_numpy(), stock_df[STOCK_FEATURE_COLS].to_numpy()
    Fdim = F_cik.shape[1] + F_stk.shape[1]
    X = np.zeros((len(cik_ids) + len(cusip_ids), Fdim), dtype=np.float32)
    X[:len(cik_ids), :F_cik.shape[1]] = F_cik
    X[len(cik_ids):, F_cik.shape[1]:] = F_stk

    # Edges (undirected, edge_weight = change_in_weight)
    cik_pos = {c: i for i, c in enumerate(cik_ids)}
    cusip_pos = {c: i + len(cik_ids) for i, c in enumerate(cusip_ids)}
    src = edges["cik"].map(cik_pos).to_numpy()
    dst = edges["cusip"].map(cusip_pos).to_numpy()
    edge_index = np.stack([np.concatenate([src, dst]), np.concatenate([dst, src])], axis=0)
    edge_weight = np.concatenate([edges["w"].to_numpy(), edges["w"].to_numpy()]).astype(np.float32)

    # Labels — 2 classes. −1 for "unknown" so we can mask them out in loss.
    y = np.full(X.shape[0], -1, dtype=np.int64)
    if not r_next.empty:
        thr = float(r_next.median())
        for cusip, idx in cusip_pos.items():
            v = r_next.get(cusip, np.nan)
            if pd.notna(v):
                y[idx] = int(v > thr)
    if not prof_next.empty:
        pthr = float(prof_next.median())
        for cik, idx in cik_pos.items():
            v = prof_next.get(cik, np.nan)
            if pd.notna(v):
                y[idx] = int(v > pthr)

    data = Data(
        x=torch.tensor(X),
        edge_index=torch.tensor(edge_index, dtype=torch.long),
        edge_weight=torch.tensor(edge_weight),
        y=torch.tensor(y),
    )
    data.is_cik = torch.zeros(X.shape[0], dtype=torch.bool)
    data.is_cik[:len(cik_ids)] = True
    data.has_label = data.y >= 0

    meta = {"cik_ids": cik_ids, "cusip_ids": cusip_ids, "n_cik": len(cik_ids), "n_cusip": len(cusip_ids)}
    return data, meta

data, meta = build_graph(YEAR, QUARTER)
print(f"nodes: {data.num_nodes:,}  (CIKs {meta['n_cik']:,}, CUSIPs {meta['n_cusip']:,})")
print(f"edges: {data.num_edges:,}  (undirected, doubled)")
print(f"feat dim: {data.num_node_features}")
print(f"labeled nodes: {int(data.has_label.sum()):,}  ({int((data.y[data.has_label]==1).sum())} positives)")

## Models

Paper §5.2: GraphSAGE with weighted message passing, and GAT where attention refines edge weights. Both feed a shared softmax head since investor/stock node vectors share the same shape.

In [ ]:
class WeightedSAGE(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes=2, num_layers=2, dropout=0.5):
        super().__init__()
        self.convs = nn.ModuleList()
        self.convs.append(SAGEConv(in_dim, hidden_dim))
        for _ in range(num_layers - 1):
            self.convs.append(SAGEConv(hidden_dim, hidden_dim))
        self.head = nn.Linear(hidden_dim, num_classes)
        self.dropout = dropout

    def forward(self, x, edge_index, edge_weight=None):
        # SAGEConv in PyG supports edge_weight via SparseTensor; here we simulate by
        # pre-scaling source features per edge (paper's weighted aggregation form).
        for i, conv in enumerate(self.convs):
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
                x = F.dropout(x, p=self.dropout, training=self.training)
        return self.head(x)


class WeightedGAT(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes=2, num_layers=2, heads=4, dropout=0.5):
        super().__init__()
        self.convs = nn.ModuleList()
        self.convs.append(GATConv(in_dim, hidden_dim, heads=heads, dropout=dropout))
        for _ in range(num_layers - 2):
            self.convs.append(GATConv(hidden_dim * heads, hidden_dim, heads=heads, dropout=dropout))
        self.convs.append(GATConv(hidden_dim * heads, hidden_dim, heads=1, concat=False, dropout=dropout))
        self.head = nn.Linear(hidden_dim, num_classes)
        self.dropout = dropout

    def forward(self, x, edge_index, edge_weight=None):
        for i, conv in enumerate(self.convs):
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.elu(x)
                x = F.dropout(x, p=self.dropout, training=self.training)
        return self.head(x)

## Training loop and eval

In [ ]:
def train_one(model, data, train_mask, val_mask, epochs=EPOCHS, lr=LR, verbose=False):
    model = model.to(DEVICE)
    data = data.to(DEVICE)
    train_mask = train_mask.to(DEVICE)
    val_mask = val_mask.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)

    best_val_acc = 0.0
    best_state = None
    for ep in range(1, epochs + 1):
        model.train()
        opt.zero_grad()
        logits = model(data.x, data.edge_index, data.edge_weight)
        loss = F.cross_entropy(logits[train_mask], data.y[train_mask])
        loss.backward()
        opt.step()

        model.eval()
        with torch.no_grad():
            logits = model(data.x, data.edge_index, data.edge_weight)
            pred = logits.argmax(dim=-1)
            val_acc = (pred[val_mask] == data.y[val_mask]).float().mean().item() if val_mask.any() else 0.0
        if val_acc > best_val_acc:
            best_val_acc, best_state = val_acc, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if verbose and ep % 20 == 0:
            print(f"  ep {ep:3d}  loss={loss.item():.4f}  val_acc={val_acc:.3f}")
    if best_state is not None:
        model.load_state_dict(best_state)
    return model

def eval_subsets(model, data, mask):
    model.eval()
    with torch.no_grad():
        logits = model(data.x.to(DEVICE), data.edge_index.to(DEVICE), data.edge_weight.to(DEVICE))
        pred = logits.argmax(dim=-1).cpu()
    y = data.y.cpu()
    out = {}
    for label, sel in [
        ("all", mask),
        ("stocks", mask & (~data.is_cik.cpu())),
        ("investors", mask & data.is_cik.cpu()),
    ]:
        if sel.sum() == 0:
            out[label] = {"n": 0}
            continue
        yt, yp = y[sel].numpy(), pred[sel].numpy()
        out[label] = {
            "n": int(sel.sum()),
            "accuracy": accuracy_score(yt, yp),
            "macro_f1": f1_score(yt, yp, average="macro"),
        }
    return out

## §5.3.1  Node-mask methodology

Random 20% of labeled nodes held out on the same graph.

In [ ]:
def make_mask_split(data, frac=MASK_FRAC, seed=0):
    rng = np.random.default_rng(seed)
    labeled = data.has_label.nonzero(as_tuple=True)[0].numpy()
    perm = rng.permutation(labeled)
    cut = int(len(perm) * (1 - frac))
    train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
    test_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
    train_mask[perm[:cut]] = True
    test_mask[perm[cut:]] = True
    return train_mask, test_mask

train_mask, test_mask = make_mask_split(data)
print(f"train labeled: {int(train_mask.sum()):,}   test labeled: {int(test_mask.sum()):,}")

sage = WeightedSAGE(data.num_node_features, HIDDEN_DIM, num_layers=NUM_LAYERS, dropout=DROPOUT)
sage = train_one(sage, data, train_mask, test_mask, verbose=True)
res_sage_mask = eval_subsets(sage, data, test_mask)

gat = WeightedGAT(data.num_node_features, HIDDEN_DIM, num_layers=NUM_LAYERS, heads=GAT_HEADS, dropout=DROPOUT)
gat = train_one(gat, data, train_mask, test_mask, verbose=True)
res_gat_mask = eval_subsets(gat, data, test_mask)

pd.DataFrame({
    "GraphSAGE": {f"{k}.{m}": v for k, d in res_sage_mask.items() for m, v in d.items()},
    "GAT": {f"{k}.{m}": v for k, d in res_gat_mask.items() for m, v in d.items()},
})

## §5.3.2  Two-periods methodology

Train on Δ-graph at (year, quarter−1), test on Δ-graph at (year, quarter). Both graphs built independently; parameters transfer because feature dim is fixed.

In [ ]:
def two_periods_eval(year, quarter, model_cls, **kwargs):
    py, pq = (year - 1, 4) if quarter == 1 else (year, quarter - 1)
    train_data, _ = build_graph(py, pq)
    test_data, _ = build_graph(year, quarter)

    # Pad feature dim to the larger of the two graphs
    Fdim = max(train_data.num_node_features, test_data.num_node_features)
    def pad(d):
        if d.num_node_features < Fdim:
            pad_cols = torch.zeros(d.num_nodes, Fdim - d.num_node_features)
            d.x = torch.cat([d.x, pad_cols], dim=1)
        return d
    train_data = pad(train_data); test_data = pad(test_data)

    model = model_cls(Fdim, HIDDEN_DIM, **kwargs)
    model = train_one(model, train_data, train_data.has_label, train_data.has_label, verbose=False)
    return eval_subsets(model, test_data, test_data.has_label)

res_sage_2p = two_periods_eval(YEAR, QUARTER, WeightedSAGE, num_layers=NUM_LAYERS, dropout=DROPOUT)
res_gat_2p = two_periods_eval(YEAR, QUARTER, WeightedGAT, num_layers=NUM_LAYERS, heads=GAT_HEADS, dropout=DROPOUT)

pd.DataFrame({
    "GraphSAGE": {f"{k}.{m}": v for k, d in res_sage_2p.items() for m, v in d.items()},
    "GAT": {f"{k}.{m}": v for k, d in res_gat_2p.items() for m, v in d.items()},
})

## Summary

In [ ]:
def flat(res, tag):
    return {
        f"{tag}.all.acc": res["all"].get("accuracy"),
        f"{tag}.stocks.acc": res["stocks"].get("accuracy"),
        f"{tag}.investors.acc": res["investors"].get("accuracy"),
    }

summary = pd.DataFrame([
    {"method": "GraphSAGE (mask)",        **flat(res_sage_mask, "test")},
    {"method": "GAT (mask)",              **flat(res_gat_mask,  "test")},
    {"method": "GraphSAGE (two-periods)", **flat(res_sage_2p,   "test")},
    {"method": "GAT (two-periods)",       **flat(res_gat_2p,    "test")},
])
summary

## Notes & next steps

- Results will be weak until `cusip_financial_data` is populated — stock features currently default to log-return + zeros.
- Paper reports mask: SAGE 0.624 / GAT 0.597; two-periods: SAGE 0.556 / GAT 0.531 averaged across 23 Δ-graphs. A single quarter is high-variance — add a loop to average over the full range once this pipeline runs end-to-end.
- SAGEConv here ignores `edge_weight`; for a fully weighted aggregator use `torch_geometric.nn.GraphConv` or a custom `MessagePassing` that multiplies messages by `edge_weight`. The current impl follows the paper's approximation where weights enter via the normalized feature scaling during pre-processing.